# 03 - Spark SQL et DataFrames - Partie 3
Lecture de `trips_history.jsonl` et `stations.json` avec un **schema explicite** (`StructType`), sans inference.

In [1]:
import sys, time
sys.path.append('/home/jovyan/work/pipeline'); sys.path.append('/home/jovyan/work/data')
from pedali_spark import get_spark
spark = get_spark('pedali-03-dataframes')
sc = spark.sparkContext
print('Spark', spark.version, '| master', sc.master, '| defaultParallelism', sc.defaultParallelism)

import df_analysis as da
from pyspark.sql import functions as F
stations, trips = da.load(spark)
trips.printSchema(); stations.printSchema()
print('evenements (dedoublonnes) :', trips.count(), '| stations :', stations.count())

Spark 3.5.3 | master spark://spark-master:7077 | defaultParallelism 4


root
 |-- trip_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- vehicle_id: string (nullable = true)
 |-- vehicle_type: string (nullable = true)
 |-- start_station: string (nullable = true)
 |-- end_station: string (nullable = true)
 |-- city: string (nullable = true)
 |-- status: string (nullable = true)
 |-- duration_s: integer (nullable = true)
 |-- distance_km: double (nullable = true)
 |-- battery_pct: integer (nullable = true)
 |-- price_mad: double (nullable = true)
 |-- event_time: timestamp (nullable = true)

root
 |-- station_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- district: string (nullable = true)
 |-- zone_type: string (nullable = true)
 |-- capacity: integer (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)



evenements (dedoublonnes) : 48997 | stations : 400


## 1. Chiffre d'affaires par ville : API DataFrame puis SQL

In [2]:
res = da.analyses(spark, stations, trips)
res['ca_df'].show()
res['ca_sql'].show()
assert res['ca_df'].collect() == res['ca_sql'].collect()
print('DataFrame == SQL : OK')

+----------+----------------+-------+
|      city|chiffre_affaires|trajets|
+----------+----------------+-------+
|Casablanca|        75430.27|   5268|
| Marrakech|        51959.36|   3619|
|     Rabat|        43473.12|   3031|
|    Tanger|        42600.26|   2962|
+----------+----------------+-------+



+----------+----------------+-------+
|      city|chiffre_affaires|trajets|
+----------+----------------+-------+
|Casablanca|        75430.27|   5268|
| Marrakech|        51959.36|   3619|
|     Rabat|        43473.12|   3031|
|    Tanger|        42600.26|   2962|
+----------+----------------+-------+



DataFrame == SQL : OK


## 2. Jointure trajets termines / referentiel stations (station de depart)
Nombre de trajets et chiffre d'affaires **par type de zone et par ville**.

In [3]:
res['zone_df'].show(30, truncate=False)
assert res['zone_df'].collect() == res['zone_sql'].collect()
print('jointure DataFrame == SQL : OK')

+----------+---------------+-------+----------------+
|city      |zone_type      |trajets|chiffre_affaires|
+----------+---------------+-------+----------------+
|Casablanca|residentiel    |1943   |27780.88        |
|Casablanca|centre_affaires|966    |13831.67        |
|Casablanca|commercial     |589    |8331.37         |
|Casablanca|tourisme       |562    |8107.99         |
|Casablanca|universite     |467    |6792.91         |
|Casablanca|plage          |465    |6638.11         |
|Casablanca|gare           |276    |3947.34         |
|Marrakech |residentiel    |1158   |16630.41        |
|Marrakech |tourisme       |882    |12729.75        |
|Marrakech |centre_affaires|609    |8619.0          |
|Marrakech |commercial     |413    |5942.02         |
|Marrakech |universite     |377    |5418.23         |
|Marrakech |gare           |180    |2619.95         |
|Rabat     |residentiel    |1419   |20233.24        |
|Rabat     |centre_affaires|662    |9578.7          |
|Rabat     |universite     |

jointure DataFrame == SQL : OK


## 3. UDF `trip_band(duration_s)` : court (< 10 min), moyen (10 a 30 min), long (> 30 min)
Puis la meme regle **sans UDF** (`when/otherwise`).

**Pourquoi la version native est plus performante** : une UDF Python est une boite noire pour Catalyst (pas d'optimisation, pas de generation de code Tungsten) et chaque ligne est serialisee JVM -> Python -> JVM. Les expressions `when/otherwise` s'executent dans la JVM, en codegen, sur le format binaire Tungsten.

In [4]:
b = da.bands(spark, res['ended'])
print(f"UDF Python : {b['udf'][0]*1000:8.1f} ms  {b['udf'][1]}")
print(f"when/otherwise : {b['native'][0]*1000:8.1f} ms  {b['native'][1]}")
print('resultats identiques :', b['identiques'])
spark.sql("SELECT trip_band(duration_s) AS band, COUNT(*) AS trajets FROM trips WHERE status='ENDED' GROUP BY 1 ORDER BY 1").show()

UDF Python :    929.1 ms  [('court', 2058), ('long', 886), ('moyen', 11936)]
when/otherwise :    331.3 ms  [('court', 2058), ('long', 886), ('moyen', 11936)]
resultats identiques : True


+-----+-------+
| band|trajets|
+-----+-------+
|court|   2058|
| long|    886|
|moyen|  11936|
+-----+-------+



## 4. Parquet nettoye dans MinIO : `s3a://lake/silver/trips_history/` partitionne par `city`

In [5]:
n = da.write_silver_history(trips)
back = spark.read.parquet(da.SILVER_HISTORY)
print('lignes ecrites :', n, '| relues :', back.count())
print('partitions :', sorted(r[0] for r in back.select('city').distinct().collect()))
back.select('trip_id', 'status', 'city', 'duration_s', 'trip_band').show(5)

lignes ecrites : 48997 | relues : 48997


partitions : ['Casablanca', 'Marrakech', 'Rabat', 'Tanger']


+----------+--------+----------+----------+---------+
|   trip_id|  status|      city|duration_s|trip_band|
+----------+--------+----------+----------+---------+
|T-00000002|   ENDED|Casablanca|      1114|    moyen|
|T-00000002|RESERVED|Casablanca|      NULL|     NULL|
|T-00000003|RESERVED|Casablanca|      NULL|     NULL|
|T-00000003|UNLOCKED|Casablanca|      NULL|     NULL|
|T-00000004|   ENDED|Casablanca|      1394|    moyen|
+----------+--------+----------+----------+---------+
only showing top 5 rows



In [6]:
spark.stop()